In [0]:
# This Cell is sued to get Secrets we have in Key Vaultss

client_id = dbutils.secrets.get(scope="kv-scope", key="db-secret-client-id-app-reg")
client_secret = dbutils.secrets.get(scope="kv-scope", key="db-secret-value-appregi")
tenant_id = dbutils.secrets.get(scope="kv-scope", key="db-secret-tenant")

# Storage account name
storage_account = "stdehealthcareanalytics"

# OAuth configs
spark.conf.set(f"fs.azure.account.auth.type.{storage_account}.dfs.core.windows.net", "OAuth")
spark.conf.set(f"fs.azure.account.oauth.provider.type.{storage_account}.dfs.core.windows.net", "org.apache.hadoop.fs.azurebfs.oauth2.ClientCredsTokenProvider")

spark.conf.set(f"fs.azure.account.oauth2.client.id.{storage_account}.dfs.core.windows.net", client_id)
spark.conf.set(f"fs.azure.account.oauth2.client.secret.{storage_account}.dfs.core.windows.net", client_secret)

spark.conf.set(
    f"fs.azure.account.oauth2.client.endpoint.{storage_account}.dfs.core.windows.net",
    f"https://login.microsoftonline.com/{tenant_id}/oauth2/token"
)

In [0]:
# SQL Server connection

sql_server = "healthcare-project-server-2026.database.windows.net"
sql_user = "username"
sql_pass = dbutils.secrets.get(scope="kv-scope", key="sql-pwd-azureportal")
sql_db = "healthcarebootcamp"

jdbc_url = f"jdbc:sqlserver://{sql_server}:1433;database={sql_db}"

connection_properties = {
    "user": sql_user,
    "password": sql_pass,
    "driver": "com.microsoft.sqlserver.jdbc.SQLServerDriver"
}

In [0]:
from pyspark.sql.functions import col

# Read FilesTable
files_df = spark.read.jdbc(
    url=jdbc_url,
    table="dbo.FilesTable",
    properties=connection_properties
)

# Filter provider files
pending_files_df = files_df.filter(
    (col("Status") == "Bronze_Processed") &
    (col("FileName").like("provider%"))
)

# Create batch list
batch_list = [row.FileName for row in pending_files_df.select("FileName").collect()]

print("Files to process:", batch_list)

# Read from Bronze
bronze_path = "abfss://bronze@stdehealthcareanalytics.dfs.core.windows.net/providers"

df = spark.read.format("parquet").load(bronze_path)

display(df.limit(10))

Files to process: []


provider_id,provider_name,department,specialization,experience_years,hospital_branch,provider_rating
PRV-15-0000001,"Dr. Montoya, Tammy",Pediatrics,ER,31,New Michellemouth General Hospital,4.8
PRV-15-0000002,"Dr. Anderson, William",Oncology,Pediatrics,25,East Amber General Hospital,4.1
NaN,"Dr. Gomez, Virginia",null,Neurology,35,Reeseshire General Hospital,2.3
PRV-15-0000004,NULL,Cardiology,null,5,Markfort General Hospital,3.8
PRV-15-0000005,"Dr. Roman, Jamie",Orthopedics,Cardiology,39,East Robertoshire General Hospital,3.5
PRV-15-0000006,NULL,Pediatrics,Cardiology,NULL,North Rachelchester General Hospital,1.6
PRV-15-0000007,"Dr. Jones, Karen",Cardiology,Cardiology,9,Christopherchester General Hospital,4.5
PRV-15-0000008,"Â Dr. Galvan, Deborah",Cardiology,ER,24,NaN,3.4
PRV-15-0000009,"Dr. Young, Samantha",ER,Oncology,23,South Steven General Hospital,2.2
PRV-15-0000010,NULL,Neurology,Orthopedics,17,Lake Cheryl General Hospital,1.8


In [0]:
from pyspark.sql.functions import col, trim, when, regexp_replace

# Trim all string columns
for column in df.columns:
    df = df.withColumn(column, trim(col(column)))

# Empty strings to NULL
for column in df.columns:
    df = df.withColumn(column, when(col(column) == "", None).otherwise(col(column)))

# Remove unwanted characters
for column in df.columns:
    df = df.withColumn(
        column,
        when(col(column).isNotNull(),
             regexp_replace(col(column).cast("string"), "Â|\t", "")
        ).otherwise(col(column))
    )

# Replace "NULL" with actual NULL
df = df.replace("NULL", None)

display(df.limit(10))

provider_id,provider_name,department,specialization,experience_years,hospital_branch,provider_rating
PRV-15-0000001,"Dr. Montoya, Tammy",Pediatrics,ER,31,New Michellemouth General Hospital,4.8
PRV-15-0000002,"Dr. Anderson, William",Oncology,Pediatrics,25,East Amber General Hospital,4.1
NaN,"Dr. Gomez, Virginia",null,Neurology,35,Reeseshire General Hospital,2.3
PRV-15-0000004,null,Cardiology,null,5,Markfort General Hospital,3.8
PRV-15-0000005,"Dr. Roman, Jamie",Orthopedics,Cardiology,39,East Robertoshire General Hospital,3.5
PRV-15-0000006,null,Pediatrics,Cardiology,null,North Rachelchester General Hospital,1.6
PRV-15-0000007,"Dr. Jones, Karen",Cardiology,Cardiology,9,Christopherchester General Hospital,4.5
PRV-15-0000008,"Dr. Galvan, Deborah",Cardiology,ER,24,NaN,3.4
PRV-15-0000009,"Dr. Young, Samantha",ER,Oncology,23,South Steven General Hospital,2.2
PRV-15-0000010,null,Neurology,Orthopedics,17,Lake Cheryl General Hospital,1.8


In [0]:
from pyspark.sql.functions import col
from delta.tables import DeltaTable
from pyspark.sql.utils import AnalysisException

print("Writing to Silver...")

# Filter + dedupe
df_valid = df.filter(col("provider_id").isNotNull())
df_valid = df_valid.dropDuplicates(["provider_id"])

silver_path = "abfss://silver@stdehealthcareanalytics.dfs.core.windows.net/provider/provider_silver"

try:
    delta_table = DeltaTable.forPath(spark, silver_path)

    delta_table.alias("target") \
        .merge(df_valid.alias("source"), "target.provider_id = source.provider_id") \
        .whenMatchedUpdateAll() \
        .whenNotMatchedInsertAll() \
        .execute()

    print("Merge complete")

except AnalysisException:
    df_valid.write.format("delta").mode("overwrite").save(silver_path)
    print("Initial load created")

Writing to Silver...
Merge complete


In [0]:
from pyspark.sql.functions import col

# Identify bad records
bad_df = df.filter(
    col("provider_id").isNull() |
    col("provider_name").isNull()
)

# Path
bad_path = "abfss://silver@stdehealthcareanalytics.dfs.core.windows.net/provider/badrecords"

# Write bad records
bad_df.write.format("delta").mode("append").save(bad_path)

display(bad_df)

provider_id,provider_name,department,specialization,experience_years,hospital_branch,provider_rating
PRV-15-0000004,null,Cardiology,null,5,Markfort General Hospital,3.8
PRV-15-0000006,null,Pediatrics,Cardiology,null,North Rachelchester General Hospital,1.6
PRV-15-0000010,null,Neurology,Orthopedics,17,Lake Cheryl General Hospital,1.8
PRV-15-0000016,null,Cardiology,Oncology,6,West Raven General Hospital,3.5
PRV-15-0000017,null,null,null,33,NaN,null
PRV-15-0000040,null,Oncology,null,2,Barrbury General Hospital,null
PRV-15-0000042,null,Pediatrics,Cardiology,26,Carrhaven General Hospital,null
PRV-15-0000044,null,Cardiology,Cardiology,35,Port Jasonbury General Hospital,4.8
PRV-15-0000045,null,Orthopedics,Orthopedics,2,East Stephaniemouth General Hospital,3.3
PRV-15-0000047,null,null,null,29,Lake Jesuston General Hospital,1.7


In [0]:
# Check Silver
spark.read.format("delta").load(
    "abfss://silver@stdehealthcareanalytics.dfs.core.windows.net/provider/provider_silver"
).display()

# Check Bad Records
spark.read.format("delta").load(
    "abfss://silver@stdehealthcareanalytics.dfs.core.windows.net/provider/badrecords"
).display()

provider_id,provider_name,department,specialization,experience_years,hospital_branch,provider_rating
NaN,"Dr. Archer, Joshua",Oncology,Oncology,25,Kristenstad General Hospital,4.8
PRV-15-0000015,"Dr. Hernandez, Jason",Orthopedics,Oncology,35,Reyeston General Hospital,3.2
PRV-15-0000030,"Dr. Nguyen, Cassandra",Neurology,ER,12,New Davidside General Hospital,3.1
PRV-15-0000043,"Dr. Young, Marissa",Oncology,Cardiology,29,Gibsonfurt General Hospital,2.1
PRV-15-0000050,"Dr. Campbell, Tammy",Cardiology,null,23,New Kenneth General Hospital,1.8
PRV-15-0000091,"Dr. Perez, Joshua",Cardiology,ER,23,Riveraburgh General Hospital,3.7
PRV-15-0000103,"Dr. Wilson, David",Orthopedics,Orthopedics,29,Johnsonland General Hospital,null
PRV-15-0000119,"Dr. Delgado, Christina",Orthopedics,Orthopedics,38,Lorraineland General Hospital,1.6
PRV-15-0000120,"Dr. Thomas, Kimberly",null,null,null,Dianeville General Hospital,3.1
PRV-15-0000138,"Dr. Lopez, Kenneth",Cardiology,Neurology,null,West Zachary General Hospital,3.4


provider_id,provider_name,department,specialization,experience_years,hospital_branch,provider_rating
PRV-15-0000004,null,Cardiology,null,5,Markfort General Hospital,3.8
PRV-15-0000006,null,Pediatrics,Cardiology,null,North Rachelchester General Hospital,1.6
PRV-15-0000010,null,Neurology,Orthopedics,17,Lake Cheryl General Hospital,1.8
PRV-15-0000016,null,Cardiology,Oncology,6,West Raven General Hospital,3.5
PRV-15-0000017,null,null,null,33,NaN,null
PRV-15-0000040,null,Oncology,null,2,Barrbury General Hospital,null
PRV-15-0000042,null,Pediatrics,Cardiology,26,Carrhaven General Hospital,null
PRV-15-0000044,null,Cardiology,Cardiology,35,Port Jasonbury General Hospital,4.8
PRV-15-0000045,null,Orthopedics,Orthopedics,2,East Stephaniemouth General Hospital,3.3
PRV-15-0000047,null,null,null,29,Lake Jesuston General Hospital,1.7
